<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/WORLD_MODEL_TOPO_CONTINUAL_BACKPROPAGATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## CASE1

In [3]:
import hashlib
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==============================================================================
# 1. 3D SPATIO-TEMPORAL WORLD MODEL LATENT BACKBONE
# ==============================================================================
class EmbodiedWorldModel(nn.Module):
    def __init__(self, latent_dim: int = 128, action_dim: int = 6, hidden_dim: int = 256):
        super().__init__()
        self.latent_dim = latent_dim
        self.in_proj = nn.Linear(latent_dim + action_dim, hidden_dim, bias=False)
        self.plastic_linear1 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.plastic_linear2 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj = nn.Linear(hidden_dim, latent_dim, bias=False)

    def forward(self, z_t: torch.Tensor, a_t: torch.Tensor) -> torch.Tensor:
        x = torch.cat([z_t, a_t], dim=-1)
        h = F.silu(self.in_proj(x))
        h1 = self.plastic_linear1(h)
        h2 = F.silu(self.plastic_linear2(h))
        h = h1 * h2
        z_next = self.out_proj(h)
        return z_next


# ==============================================================================
# 2. TOPOLOGICAL GOVERNOR (COORDINATE ANCHORS & GRADIENT PROJECTION)
# ==============================================================================
class TopologicalGovernor:
    def __init__(self, target_tensor: torch.nn.Parameter, prime_anchors=(2, 3, 5, 7, 11, 13)):
        self.target = target_tensor
        self.prime_anchors = torch.tensor(prime_anchors, dtype=torch.long)
        with torch.no_grad():
            self.invariant_bedrock = self.target[self.prime_anchors, :].clone().detach()

    def project_gradients(self):
        if self.target.grad is not None:
            with torch.no_grad():
                self.target.grad[self.prime_anchors, :] = 0.0

    def enforce_invariance(self):
        with torch.no_grad():
            self.target[self.prime_anchors, :] = self.invariant_bedrock

    def compute_drift(self) -> float:
        with torch.no_grad():
            current = self.target[self.prime_anchors, :]
            drift = torch.norm(current - self.invariant_bedrock).item()
            return drift

    def get_hash(self) -> str:
        raw_bytes = self.target[self.prime_anchors, :].detach().cpu().numpy().tobytes()
        return hashlib.sha256(raw_bytes).hexdigest()[:16]


# ==============================================================================
# 3. CONTINUAL BACKPROPAGATION (METABOLIC UTILITY & NEUROGENESIS)
# ==============================================================================
class ContinualPlasticityEngine:
    def __init__(self, layer1: nn.Linear, layer2: nn.Linear, decay: float = 0.99):
        self.layer1 = layer1
        self.layer2 = layer2
        self.decay = decay
        self.hidden_dim = layer1.out_features
        self.running_utility = torch.ones(self.hidden_dim)

    def update_utility(self, hidden_acts: torch.Tensor):
        with torch.no_grad():
            batch_utility = hidden_acts.abs().mean(dim=0).detach().cpu()
            self.running_utility = self.decay * self.running_utility + (1.0 - self.decay) * batch_utility

    def trigger_neurogenesis(self, reinit_fraction: float = 0.04):
        with torch.no_grad():
            num_units = int(self.hidden_dim * reinit_fraction)
            if num_units == 0:
                return 0
            _, dead_indices = torch.topk(self.running_utility, k=num_units, largest=False)
            device = self.layer1.weight.device
            dead_idx_tensor = dead_indices.to(device)
            nn.init.kaiming_normal_(self.layer1.weight[dead_idx_tensor, :])
            self.layer2.weight[:, dead_idx_tensor] = 0.0
            self.running_utility[dead_indices] = self.running_utility.mean()
            return num_units


# ==============================================================================
# 4. LONG-HORIZON SIMULATION HARNESS (10,000 STEPS)
# ==============================================================================
def run_topocbp_world_model_10k():
    torch.manual_seed(123)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[*] Initializing Topo-CBP World Model on device: {device}")

    latent_dim = 128
    action_dim = 6
    hidden_dim = 256
    batch_size = 32

    model = EmbodiedWorldModel(latent_dim=latent_dim, action_dim=action_dim, hidden_dim=hidden_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

    governor = TopologicalGovernor(model.in_proj.weight, prime_anchors=(2, 3, 5, 7, 11, 13))
    plasticity_engine = ContinualPlasticityEngine(model.plastic_linear1, model.plastic_linear2)

    initial_hash = governor.get_hash()
    print(f"[*] Baseline Invariant Hash : {initial_hash}")
    print(f"[*] Prime Anchors Activated : {governor.prime_anchors.tolist()}\n")

    print("=======================================================================================")
    print("STEP  | LOSS    | ANCHOR DRIFT    | SHA-256 HASH     | RECYCLED | REGIME")
    print("=======================================================================================")

    log_checkpoints = {
        1, 1000, 2500, 2501, 3500, 5000, 5001, 6000, 7500, 7501, 8500, 10000
    }

    for step in range(1, 10001):
        optimizer.zero_grad()

        # Multi-regime physical transitions
        if step <= 2500:
            regime = "Regime A (Newtonian)"
        elif step <= 5000:
            regime = "Regime B (Turbulence)"
        elif step <= 7500:
            regime = "Regime C (Viscous Drag)"
        else:
            regime = "Regime D (Gyroscopic)"

        z_t = torch.randn(batch_size, latent_dim, device=device)
        a_t = torch.randn(batch_size, action_dim, device=device)
        a_embedded = F.pad(a_t, (0, latent_dim - action_dim))

        # Distinct physical mapping functions
        if step <= 2500:
            target_next_z = torch.sin(z_t) + 0.1 * a_embedded
        elif step <= 5000:
            target_next_z = torch.cos(z_t * 1.5) - 0.3 * torch.tanh(a_embedded)
        elif step <= 7500:
            target_next_z = torch.sigmoid(z_t) * 1.2 + 0.05 * (a_embedded ** 2)
        else:
            target_next_z = torch.roll(z_t, shifts=1, dims=-1) * 0.8 + 0.2 * torch.sin(a_embedded)

        z_pred = model(z_t, a_t)
        loss = F.mse_loss(z_pred, target_next_z)
        loss.backward()

        governor.project_gradients()
        optimizer.step()
        governor.enforce_invariance()

        with torch.no_grad():
            h_inter = model.in_proj(torch.cat([z_t, a_t], dim=-1))
            plasticity_engine.update_utility(h_inter)

        recycled_count = 0
        if step % 100 == 0:
            recycled_count = plasticity_engine.trigger_neurogenesis(reinit_fraction=0.04)

        if step in log_checkpoints:
            drift = governor.compute_drift()
            curr_hash = governor.get_hash()
            print(f"{step:5d} | {loss.item():.5f} | {drift:.10f} | {curr_hash} | {recycled_count:8d} | {regime}")

    final_hash = governor.get_hash()
    print("=======================================================================================")
    print(f"[*] Long-Horizon Verification (10,000 Steps):")
    print(f"    - Baseline Hash: {initial_hash}")
    print(f"    - Final Hash   : {final_hash}")
    print(f"    - Invariance   : {'STRICTLY INVARIANT (PASS)' if initial_hash == final_hash else 'DRIFT DETECTED (FAIL)'}")
    print(f"    - Final Drift  : {governor.compute_drift():.10f}")


if __name__ == "__main__":
    run_topocbp_world_model_10k()

[*] Initializing Topo-CBP World Model on device: cuda
[*] Baseline Invariant Hash : 2ce28ffcf131d5c9
[*] Prime Anchors Activated : [2, 3, 5, 7, 11, 13]

STEP  | LOSS    | ANCHOR DRIFT    | SHA-256 HASH     | RECYCLED | REGIME
    1 | 0.43655 | 0.0000000000 | 2ce28ffcf131d5c9 |        0 | Regime A (Newtonian)
 1000 | 0.11825 | 0.0000000000 | 2ce28ffcf131d5c9 |       10 | Regime A (Newtonian)
 2500 | 0.09627 | 0.0000000000 | 2ce28ffcf131d5c9 |       10 | Regime A (Newtonian)
 2501 | 0.79960 | 0.0000000000 | 2ce28ffcf131d5c9 |        0 | Regime B (Turbulence)
 3500 | 0.39871 | 0.0000000000 | 2ce28ffcf131d5c9 |       10 | Regime B (Turbulence)
 5000 | 0.25089 | 0.0000000000 | 2ce28ffcf131d5c9 |       10 | Regime B (Turbulence)
 5001 | 0.26641 | 0.0000000000 | 2ce28ffcf131d5c9 |        0 | Regime C (Viscous Drag)
 6000 | 0.00721 | 0.0000000000 | 2ce28ffcf131d5c9 |       10 | Regime C (Viscous Drag)
 7500 | 0.00552 | 0.0000000000 | 2ce28ffcf131d5c9 |       10 | Regime C (Viscous Drag)
 7501 

## CASE2

In [4]:
import hashlib
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==============================================================================
# 1. 3D SPATIO-TEMPORAL WORLD MODEL LATENT BACKBONE
# ==============================================================================
class EmbodiedWorldModel(nn.Module):
    def __init__(self, latent_dim: int = 128, action_dim: int = 6, hidden_dim: int = 256):
        super().__init__()
        self.latent_dim = latent_dim
        self.in_proj = nn.Linear(latent_dim + action_dim, hidden_dim, bias=False)
        self.plastic_linear1 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.plastic_linear2 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj = nn.Linear(hidden_dim, latent_dim, bias=False)

    def forward(self, z_t: torch.Tensor, a_t: torch.Tensor) -> torch.Tensor:
        x = torch.cat([z_t, a_t], dim=-1)
        h = F.silu(self.in_proj(x))
        h1 = self.plastic_linear1(h)
        h2 = F.silu(self.plastic_linear2(h))
        h = h1 * h2
        z_next = self.out_proj(h)
        return z_next


# ==============================================================================
# 2. TOPOLOGICAL GOVERNOR (COORDINATE ANCHORS & GRADIENT PROJECTION)
# ==============================================================================
class TopologicalGovernor:
    def __init__(self, target_tensor: torch.nn.Parameter, prime_anchors=(2, 3, 5, 7, 11, 13)):
        self.target = target_tensor
        self.prime_anchors = torch.tensor(prime_anchors, dtype=torch.long)
        with torch.no_grad():
            self.invariant_bedrock = self.target[self.prime_anchors, :].clone().detach()

    def project_gradients(self):
        if self.target.grad is not None:
            with torch.no_grad():
                self.target.grad[self.prime_anchors, :] = 0.0

    def enforce_invariance(self):
        with torch.no_grad():
            self.target[self.prime_anchors, :] = self.invariant_bedrock

    def compute_drift(self) -> float:
        with torch.no_grad():
            current = self.target[self.prime_anchors, :]
            drift = torch.norm(current - self.invariant_bedrock).item()
            return drift

    def get_hash(self) -> str:
        raw_bytes = self.target[self.prime_anchors, :].detach().cpu().numpy().tobytes()
        return hashlib.sha256(raw_bytes).hexdigest()[:16]


# ==============================================================================
# 3. CONTINUAL BACKPROPAGATION (METABOLIC UTILITY & NEUROGENESIS)
# ==============================================================================
class ContinualPlasticityEngine:
    def __init__(self, layer1: nn.Linear, layer2: nn.Linear, decay: float = 0.99):
        self.layer1 = layer1
        self.layer2 = layer2
        self.decay = decay
        self.hidden_dim = layer1.out_features
        self.running_utility = torch.ones(self.hidden_dim)

    def update_utility(self, hidden_acts: torch.Tensor):
        with torch.no_grad():
            batch_utility = hidden_acts.abs().mean(dim=0).detach().cpu()
            self.running_utility = self.decay * self.running_utility + (1.0 - self.decay) * batch_utility

    def trigger_neurogenesis(self, reinit_fraction: float = 0.04):
        with torch.no_grad():
            num_units = int(self.hidden_dim * reinit_fraction)
            if num_units == 0:
                return 0
            _, dead_indices = torch.topk(self.running_utility, k=num_units, largest=False)
            device = self.layer1.weight.device
            dead_idx_tensor = dead_indices.to(device)
            nn.init.kaiming_normal_(self.layer1.weight[dead_idx_tensor, :])
            self.layer2.weight[:, dead_idx_tensor] = 0.0
            self.running_utility[dead_indices] = self.running_utility.mean()
            return num_units


# ==============================================================================
# 4. SIDE-BY-SIDE SIMULATION: BASELINE (NO TOPO) VS. TOPO-CBP (10,000 STEPS)
# ==============================================================================
def run_comparative_world_model_simulation():
    torch.manual_seed(123)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[*] Initializing Comparative World Model Harness on device: {device}")

    latent_dim = 128
    action_dim = 6
    hidden_dim = 256
    batch_size = 32
    prime_anchors = (2, 3, 5, 7, 11, 13)

    # 1. Instantiate Baseline Model (CBP Only, Unconstrained Anchors)
    torch.manual_seed(123)
    model_baseline = EmbodiedWorldModel(latent_dim=latent_dim, action_dim=action_dim, hidden_dim=hidden_dim).to(device)
    opt_baseline = torch.optim.AdamW(model_baseline.parameters(), lr=1e-3, weight_decay=1e-4)
    cbp_baseline = ContinualPlasticityEngine(model_baseline.plastic_linear1, model_baseline.plastic_linear2)
    # Observer for baseline drift (measures drift without intervening)
    baseline_observer = TopologicalGovernor(model_baseline.in_proj.weight, prime_anchors=prime_anchors)

    # 2. Instantiate Governed Model (Topo-CBP)
    torch.manual_seed(123)
    model_topo = EmbodiedWorldModel(latent_dim=latent_dim, action_dim=action_dim, hidden_dim=hidden_dim).to(device)
    opt_topo = torch.optim.AdamW(model_topo.parameters(), lr=1e-3, weight_decay=1e-4)
    cbp_topo = ContinualPlasticityEngine(model_topo.plastic_linear1, model_topo.plastic_linear2)
    topo_governor = TopologicalGovernor(model_topo.in_proj.weight, prime_anchors=prime_anchors)

    initial_hash = topo_governor.get_hash()
    print(f"[*] Baseline Invariant Hash : {initial_hash}")
    print(f"[*] Prime Anchors Activated : {list(prime_anchors)}\n")

    print("=" * 105)
    print(f"{'STEP':>5} | {'BASE LOSS':>9} | {'BASE DRIFT':>10} | {'TOPO LOSS':>9} | {'TOPO DRIFT':>12} | {'TOPO SHA-256':>16} | {'REGIME'}")
    print("=" * 105)

    log_checkpoints = {
        1, 1000, 2500, 2501, 3500, 5000, 5001, 6000, 7500, 7501, 8500, 10000
    }

    for step in range(1, 10001):
        # Multi-regime physical transitions
        if step <= 2500:
            regime = "Regime A (Newtonian)"
        elif step <= 5000:
            regime = "Regime B (Turbulence)"
        elif step <= 7500:
            regime = "Regime C (Viscous Drag)"
        else:
            regime = "Regime D (Gyroscopic)"

        # Input tensors
        z_t = torch.randn(batch_size, latent_dim, device=device)
        a_t = torch.randn(batch_size, action_dim, device=device)
        a_embedded = F.pad(a_t, (0, latent_dim - action_dim))

        # Target dynamics
        if step <= 2500:
            target_next_z = torch.sin(z_t) + 0.1 * a_embedded
        elif step <= 5000:
            target_next_z = torch.cos(z_t * 1.5) - 0.3 * torch.tanh(a_embedded)
        elif step <= 7500:
            target_next_z = torch.sigmoid(z_t) * 1.2 + 0.05 * (a_embedded ** 2)
        else:
            target_next_z = torch.roll(z_t, shifts=1, dims=-1) * 0.8 + 0.2 * torch.sin(a_embedded)

        # ----------------------------------------------------------------------
        # BASELINE EXECUTION (CBP WITHOUT TOPOLOGICAL GOVERNOR)
        # ----------------------------------------------------------------------
        opt_baseline.zero_grad()
        z_pred_base = model_baseline(z_t, a_t)
        loss_base = F.mse_loss(z_pred_base, target_next_z)
        loss_base.backward()
        # Normal step without projection or clamping
        opt_baseline.step()

        with torch.no_grad():
            h_base = model_baseline.in_proj(torch.cat([z_t, a_t], dim=-1))
            cbp_baseline.update_utility(h_base)
        if step % 100 == 0:
            cbp_baseline.trigger_neurogenesis(reinit_fraction=0.04)

        # ----------------------------------------------------------------------
        # TOPO-CBP EXECUTION (CBP WITH TOPOLOGICAL GOVERNOR)
        # ----------------------------------------------------------------------
        opt_topo.zero_grad()
        z_pred_topo = model_topo(z_t, a_t)
        loss_topo = F.mse_loss(z_pred_topo, target_next_z)
        loss_topo.backward()

        # Topological step: orthogonal gradient projection
        topo_governor.project_gradients()
        opt_topo.step()
        # Topological step: bitwise invariance restoration
        topo_governor.enforce_invariance()

        with torch.no_grad():
            h_topo = model_topo.in_proj(torch.cat([z_t, a_t], dim=-1))
            cbp_topo.update_utility(h_topo)
        if step % 100 == 0:
            cbp_topo.trigger_neurogenesis(reinit_fraction=0.04)

        # ----------------------------------------------------------------------
        # TELEMETRY LOGGING
        # ----------------------------------------------------------------------
        if step in log_checkpoints:
            drift_base = baseline_observer.compute_drift()
            drift_topo = topo_governor.compute_drift()
            hash_topo = topo_governor.get_hash()
            print(f"{step:5d} | {loss_base.item():9.5f} | {drift_base:10.5f} | {loss_topo.item():9.5f} | {drift_topo:12.10f} | {hash_topo:16} | {regime}")

    print("=" * 105)
    print(f"[*] Post-Simulation Verification Summary:")
    print(f"    - Baseline Drift (Without Topo) : {baseline_observer.compute_drift():.6f}")
    print(f"    - Baseline Invariant Hash       : {baseline_observer.get_hash()} (Mutated)")
    print(f"    - Topo-CBP Drift (With Topo)    : {topo_governor.compute_drift():.10f}")
    print(f"    - Topo-CBP Invariant Hash       : {topo_governor.get_hash()} (Locked)")


if __name__ == "__main__":
    run_comparative_world_model_simulation()

[*] Initializing Comparative World Model Harness on device: cuda
[*] Baseline Invariant Hash : 2ce28ffcf131d5c9
[*] Prime Anchors Activated : [2, 3, 5, 7, 11, 13]

 STEP | BASE LOSS | BASE DRIFT | TOPO LOSS |   TOPO DRIFT |     TOPO SHA-256 | REGIME
    1 |   0.43655 |    0.02831 |   0.43655 | 0.0000000000 | 2ce28ffcf131d5c9 | Regime A (Newtonian)
 1000 |   0.10885 |    0.99194 |   0.10912 | 0.0000000000 | 2ce28ffcf131d5c9 | Regime A (Newtonian)
 2500 |   0.10340 |    0.95209 |   0.10440 | 0.0000000000 | 2ce28ffcf131d5c9 | Regime A (Newtonian)
 2501 |   0.81384 |    0.95108 |   0.81159 | 0.0000000000 | 2ce28ffcf131d5c9 | Regime B (Turbulence)
 3500 |   0.40796 |    1.06426 |   0.40770 | 0.0000000000 | 2ce28ffcf131d5c9 | Regime B (Turbulence)
 5000 |   0.25106 |    1.45112 |   0.26162 | 0.0000000000 | 2ce28ffcf131d5c9 | Regime B (Turbulence)
 5001 |   0.25707 |    1.45044 |   0.25949 | 0.0000000000 | 2ce28ffcf131d5c9 | Regime C (Viscous Drag)
 6000 |   0.00694 |    1.21206 |   0.00715 |

## CASE3

In [5]:
import hashlib
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==============================================================================
# 1. 3D SPATIO-TEMPORAL WORLD MODEL LATENT BACKBONE
# ==============================================================================
class EmbodiedWorldModel(nn.Module):
    def __init__(self, latent_dim: int = 128, action_dim: int = 6, hidden_dim: int = 256):
        super().__init__()
        self.latent_dim = latent_dim
        self.in_proj = nn.Linear(latent_dim + action_dim, hidden_dim, bias=False)
        self.plastic_linear1 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.plastic_linear2 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj = nn.Linear(hidden_dim, latent_dim, bias=False)

    def forward(self, z_t: torch.Tensor, a_t: torch.Tensor) -> torch.Tensor:
        x = torch.cat([z_t, a_t], dim=-1)
        h = F.silu(self.in_proj(x))
        h1 = self.plastic_linear1(h)
        h2 = F.silu(self.plastic_linear2(h))
        h = h1 * h2
        z_next = self.out_proj(h)
        return z_next


# ==============================================================================
# 2. TOPOLOGICAL GOVERNOR (COORDINATE ANCHORS & GRADIENT PROJECTION)
# ==============================================================================
class TopologicalGovernor:
    def __init__(self, target_tensor: torch.nn.Parameter, prime_anchors=(2, 3, 5, 7, 11, 13)):
        self.target = target_tensor
        self.prime_anchors = torch.tensor(prime_anchors, dtype=torch.long)
        with torch.no_grad():
            self.invariant_bedrock = self.target[self.prime_anchors, :].clone().detach()

    def project_gradients(self):
        if self.target.grad is not None:
            with torch.no_grad():
                self.target.grad[self.prime_anchors, :] = 0.0

    def enforce_invariance(self):
        with torch.no_grad():
            self.target[self.prime_anchors, :] = self.invariant_bedrock

    def compute_drift(self) -> float:
        with torch.no_grad():
            current = self.target[self.prime_anchors, :]
            drift = torch.norm(current - self.invariant_bedrock).item()
            return drift

    def get_hash(self) -> str:
        raw_bytes = self.target[self.prime_anchors, :].detach().cpu().numpy().tobytes()
        return hashlib.sha256(raw_bytes).hexdigest()[:16]


# ==============================================================================
# 3. CONTINUAL BACKPROPAGATION (METABOLIC UTILITY & NEUROGENESIS)
# ==============================================================================
class ContinualPlasticityEngine:
    def __init__(self, layer1: nn.Linear, layer2: nn.Linear, decay: float = 0.99):
        self.layer1 = layer1
        self.layer2 = layer2
        self.decay = decay
        self.hidden_dim = layer1.out_features
        self.running_utility = torch.ones(self.hidden_dim)

    def update_utility(self, hidden_acts: torch.Tensor):
        with torch.no_grad():
            batch_utility = hidden_acts.abs().mean(dim=0).detach().cpu()
            self.running_utility = self.decay * self.running_utility + (1.0 - self.decay) * batch_utility

    def trigger_neurogenesis(self, reinit_fraction: float = 0.04):
        with torch.no_grad():
            num_units = int(self.hidden_dim * reinit_fraction)
            if num_units == 0:
                return 0
            _, dead_indices = torch.topk(self.running_utility, k=num_units, largest=False)
            device = self.layer1.weight.device
            dead_idx_tensor = dead_indices.to(device)
            nn.init.kaiming_normal_(self.layer1.weight[dead_idx_tensor, :])
            self.layer2.weight[:, dead_idx_tensor] = 0.0
            self.running_utility[dead_indices] = self.running_utility.mean()
            return num_units


# ==============================================================================
# 4. ULTRA LONG-HORIZON SIMULATION HARNESS (100,000 STEPS)
# ==============================================================================
def run_100k_comparative_simulation():
    torch.manual_seed(123)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[*] Initializing Ultra Long-Horizon (100,000 Steps) Harness on: {device}")

    latent_dim = 128
    action_dim = 6
    hidden_dim = 256
    batch_size = 32
    prime_anchors = (2, 3, 5, 7, 11, 13)

    # 1. Baseline Model (CBP Only, Free Drifting Anchors)
    torch.manual_seed(123)
    model_baseline = EmbodiedWorldModel(latent_dim=latent_dim, action_dim=action_dim, hidden_dim=hidden_dim).to(device)
    opt_baseline = torch.optim.AdamW(model_baseline.parameters(), lr=1e-3, weight_decay=1e-4)
    cbp_baseline = ContinualPlasticityEngine(model_baseline.plastic_linear1, model_baseline.plastic_linear2)
    baseline_observer = TopologicalGovernor(model_baseline.in_proj.weight, prime_anchors=prime_anchors)

    # 2. Governed Model (Topo-CBP)
    torch.manual_seed(123)
    model_topo = EmbodiedWorldModel(latent_dim=latent_dim, action_dim=action_dim, hidden_dim=hidden_dim).to(device)
    opt_topo = torch.optim.AdamW(model_topo.parameters(), lr=1e-3, weight_decay=1e-4)
    cbp_topo = ContinualPlasticityEngine(model_topo.plastic_linear1, model_topo.plastic_linear2)
    topo_governor = TopologicalGovernor(model_topo.in_proj.weight, prime_anchors=prime_anchors)

    initial_hash = topo_governor.get_hash()
    print(f"[*] Baseline Invariant Hash : {initial_hash}")
    print(f"[*] Prime Anchors Activated : {list(prime_anchors)}\n")

    print("=" * 110)
    print(f"{'STEP':>6} | {'BASE LOSS':>9} | {'BASE DRIFT':>10} | {'TOPO LOSS':>9} | {'TOPO DRIFT':>12} | {'TOPO SHA-256':>16} | {'REGIME'}")
    print("=" * 110)

    log_checkpoints = {
        1, 10000, 25000, 25001, 35000, 50000, 50001, 60000, 75000, 75001, 85000, 100000
    }

    t0 = time.time()

    for step in range(1, 100001):
        # 4 Broad Regimes across 100,000 steps
        if step <= 25000:
            regime = "Regime A (Newtonian)"
        elif step <= 50000:
            regime = "Regime B (Turbulence)"
        elif step <= 75000:
            regime = "Regime C (Viscous Drag)"
        else:
            regime = "Regime D (Gyroscopic)"

        z_t = torch.randn(batch_size, latent_dim, device=device)
        a_t = torch.randn(batch_size, action_dim, device=device)
        a_embedded = F.pad(a_t, (0, latent_dim - action_dim))

        # Target dynamics
        if step <= 25000:
            target_next_z = torch.sin(z_t) + 0.1 * a_embedded
        elif step <= 50000:
            target_next_z = torch.cos(z_t * 1.5) - 0.3 * torch.tanh(a_embedded)
        elif step <= 75000:
            target_next_z = torch.sigmoid(z_t) * 1.2 + 0.05 * (a_embedded ** 2)
        else:
            target_next_z = torch.roll(z_t, shifts=1, dims=-1) * 0.8 + 0.2 * torch.sin(a_embedded)

        # Baseline Execution
        opt_baseline.zero_grad()
        z_pred_base = model_baseline(z_t, a_t)
        loss_base = F.mse_loss(z_pred_base, target_next_z)
        loss_base.backward()
        opt_baseline.step()

        with torch.no_grad():
            h_base = model_baseline.in_proj(torch.cat([z_t, a_t], dim=-1))
            cbp_baseline.update_utility(h_base)
        if step % 100 == 0:
            cbp_baseline.trigger_neurogenesis(reinit_fraction=0.04)

        # Topo-CBP Execution
        opt_topo.zero_grad()
        z_pred_topo = model_topo(z_t, a_t)
        loss_topo = F.mse_loss(z_pred_topo, target_next_z)
        loss_topo.backward()
        topo_governor.project_gradients()
        opt_topo.step()
        topo_governor.enforce_invariance()

        with torch.no_grad():
            h_topo = model_topo.in_proj(torch.cat([z_t, a_t], dim=-1))
            cbp_topo.update_utility(h_topo)
        if step % 100 == 0:
            cbp_topo.trigger_neurogenesis(reinit_fraction=0.04)

        # Telemetry
        if step in log_checkpoints:
            drift_base = baseline_observer.compute_drift()
            drift_topo = topo_governor.compute_drift()
            hash_topo = topo_governor.get_hash()
            print(f"{step:6d} | {loss_base.item():9.5f} | {drift_base:10.5f} | {loss_topo.item():9.5f} | {drift_topo:12.10f} | {hash_topo:16} | {regime}")

    elapsed = time.time() - t0
    print("=" * 110)
    print(f"[*] Completed 100,000 steps in {elapsed:.2f}s ({100000/elapsed:.1f} steps/s)")
    print(f"[*] Ultra Long-Horizon Verification Summary:")
    print(f"    - Baseline Drift (Without Topo) : {baseline_observer.compute_drift():.6f}")
    print(f"    - Baseline Invariant Hash       : {baseline_observer.get_hash()} (Mutated)")
    print(f"    - Topo-CBP Drift (With Topo)    : {topo_governor.compute_drift():.10f}")
    print(f"    - Topo-CBP Invariant Hash       : {topo_governor.get_hash()} (Locked)")


if __name__ == "__main__":
    run_100k_comparative_simulation()

[*] Initializing Ultra Long-Horizon (100,000 Steps) Harness on: cuda
[*] Baseline Invariant Hash : 2ce28ffcf131d5c9
[*] Prime Anchors Activated : [2, 3, 5, 7, 11, 13]

  STEP | BASE LOSS | BASE DRIFT | TOPO LOSS |   TOPO DRIFT |     TOPO SHA-256 | REGIME
     1 |   0.43655 |    0.02831 |   0.43655 | 0.0000000000 | 2ce28ffcf131d5c9 | Regime A (Newtonian)
 10000 |   0.08454 |    1.29621 |   0.08560 | 0.0000000000 | 2ce28ffcf131d5c9 | Regime A (Newtonian)
 25000 |   0.06789 |    1.34076 |   0.06845 | 0.0000000000 | 2ce28ffcf131d5c9 | Regime A (Newtonian)
 25001 |   0.89798 |    1.34070 |   0.89202 | 0.0000000000 | 2ce28ffcf131d5c9 | Regime B (Turbulence)
 35000 |   0.13930 |    2.23976 |   0.17404 | 0.0000000000 | 2ce28ffcf131d5c9 | Regime B (Turbulence)
 50000 |   0.11691 |    2.39186 |   0.12974 | 0.0000000000 | 2ce28ffcf131d5c9 | Regime B (Turbulence)
 50001 |   0.43232 |    2.39086 |   0.43064 | 0.0000000000 | 2ce28ffcf131d5c9 | Regime C (Viscous Drag)
 60000 |   0.00526 |    2.28991 